# 3.3 数据排布格式：BSND / BNSD / BSH / TND

FA 算子接口里的 `inputLayout` 参数决定 Q、K、V 张量在内存里怎么摆。排布不是小事——它决定了「取一个计算基本块」时地址是**顺序的**还是**跳跃的**，直接影响搬运效率。

## 1. 四个字母的含义

| 字母 | 含义 |
|--|--|
| B（Batch） | 批处理大小 |
| S（Seq-Length） | 序列长度 |
| N（Head-Num） | 多头数 |
| D（Head-Dim） | 单头向量维度，满足 H = N × D |

同一份 Q 张量（B=2, N=4, S=256, D=128），四种排布的摆放方式：

![四种数据排布](images/data_layout_formats.svg)

## 2. BSND 与 BSH：token 优先

**BSND**（多头注意力的常见输入）按 **D → N → S → B** 的顺序排布：先存完一个 D 维向量，接着排同一 token 的下一个头，排满 N×D 后才是下一个 token。

> **特点：每个 token 的多个头连续存储，同一个头内的不同 token 不连续。**

当 S 维各 batch 长度不一时，BSND（以及 BSH）会自动 **padding 对齐**到最长 S——短序列的补齐区就是白白占用的内存。

**BSH** 是单头场景的常见格式：把 N、D 合成 H = N×D，shape 变成 `[B, S, H]`。存储顺序与 BSND 一致，同样存在 padding 浪费。

基本块视角（设基本块大小 `(s1, d) = (128, 128)`，BNSD=(2,4,256,128) 的张量切成 2×4×2 = 16 个基本块）：BSND 下**一个基本块内的地址是跳跃的**，取数需要按 stride 计算，每个 token 的头之间隔着 N×D。

## 3. BNSD：头优先

**BNSD** 按 **D → S → N → B** 排布：同一个头内的所有 token 连续存放，一个头排完才是下一个头。

> **特点：同一个头内的不同 token 连续，不同头之间不连续。**

同样切 16 个基本块，BNSD 下每个基本块 `[128, 128]` 在张量内**内存连续**，取基本块就是顺序取——这就是「计算偏移时是顺序的」的含义。对以头为单位分核计算的 FA 来说，BNSD 是对搬运最友好的排布。

## 4. TND：合轴去 padding

BSND 因 padding 存在内存浪费，于是把 B、S 两维**合成一个 T 轴**：`TND = [B×S, N, D]`。

- **优点**：各 batch 序列长度不同也无需 padding，**unpadding 节省内存**。
- **代价**：T 轴切基本块会产生**尾块**——基本块 `128×128` 切下来，有些块的 S 维只有 64 或 72；且基本块在内存中不再整齐对齐，取数同样需要跳址。

### actualSeqLengths 的两种语义

| 排布 | `actualSeqLengths` 含义 | 示例 |
|--|--|--|
| BSND / BNSD / BSH | 每条序列的**实际长度**（逐条给出） | `[8, 6, 4]` |
| TND | **叠加长度**，第 i 条 = 第 i 项 − 第 i−1 项 | `[8, 14, 18]` → 三条序列分别为 8、6、4 |

KV 侧还有对应的 `actualSeqLengthsKv`，语义相同。

## 5. Q 与 KV 的 S 可以不同

以上三种格式都支持 **Q 的 S（记 S1）≠ KV 的 S（记 S2）**，对应三类真实场景：

| 场景 | S1 与 S2 的关系 |
|--|--|
| 推理 **Prefill** 阶段 | S1 = S2（整段 prompt 一次性算） |
| 推理 **Decode** 阶段 | S1 = 1 ≪ S2（新 token 查询全部 KV Cache） |
| 稀疏场景 | S1 > S2 |

> 这是第 2 章 Prefill/Decode 知识在接口层的落点：Decode 时 query 只有 1 个 token，key/value 却是几百上千。

## 6. GQA 下的排布

GQA：一组 Q 共享同一份 KV。设 Q 头数 N1、KV 头数 N2，组数 **G = N1 / N2**，比常规多出一个 G 维。

- 存储排布：`BSNGD`（N2 个 KV 组，每组 G 个 Q 头）
- 计算排布：**按 B、N2 分核**，因此 Kernel 视角实际是 `BN2SGD`——同一份数据，按「计算视角」重排

TND 在 GQA 场景下同样将 BS 合轴，区别只是每个 batch 的 S 长度可以不同。

## 7. 基本块切分与分核

FA 算子会对 Q 切块并分核（以 PFA 为例）：假设 S 外层分块 `souter = 128`，基本块大小为 `[souter, D] = [128, 128]`。

以 `TND = [456, 4, 128]`、`actualSeqLengths = [256, 456]` 的 Q 为例：

- 全切下来共 16 个基本块；
- **4 个是尾块**（大小 `[72, 128]` 而非 `[128, 128]`）；
- 每个基本块在张量内不连续，取数要跳着取。

切完的基本块再按**负载均衡策略**（见 3.6 节）分派到不同核上执行——基本块是 FA 多核并行的最小任务单元。

## 8. 本节小结

| 排布 | 存储顺序 | 基本块取数 | 内存效率 |
|--|--|--|--|
| BSND | D→N→S→B | 跳址 | 有 padding 浪费 |
| BSH | H→S→B（单头） | 跳址 | 有 padding 浪费 |
| BNSD | D→S→N→B | **顺序** | 有 padding 浪费 |
| TND | D→N→T | 跳址 + **尾块** | **无 padding** |

- 排布决定「取基本块是否连续」，直接影响搬运指令的组织。
- `actualSeqLengths` 在 TND 下是叠加式，换算序列长度要做差。
- GQA 场景存储按 BSNGD，Kernel 计算视角重排为 BN2SGD。

## 小测验

1. 同一份 Q（B=2, N=4, S=256, D=128），BNSD 和 BSND 排布下取一个 `[128, 128]` 基本块，哪种是内存连续的？为什么？
2. TND 排布的 `actualSeqLengths = [128, 192, 224]`，三条序列各多长？相比 BSND padding 对齐省了多少 token 位置（假设三 batch 都存满 TND 即可）？
3. 为什么 TND 会产生尾块？尾块对 Kernel 计算会带来什么额外开销？